In [1]:
#task(1)
import pandas as pd

# 1. Load the dataset
df = pd.read_csv("Walmart_Sales.csv")

print("--- 1. Basic Information & Data Types ---")
print(df.info())

print("\n--- 2. Checking for Explicit Missing Values (NaN) ---")
print(df.isnull().sum())

print("\n--- 3. Checking Data Range & Scale Disparities ---")
# Transposing the describe table to easily compare min, max, and scale variances
print(df.describe().T[['min', 'mean', 'max', 'std']])

print("\n--- 4. Checking for Duplicate Rows ---")
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows found: {duplicate_count}")

print("\n--- 5. Checking for Structural Anomalies (Unique values in 'Store') ---")
print(f"Number of unique stores: {df['Store'].nunique()}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")

--- 1. Basic Information & Data Types ---
<class 'pandas.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Date          6435 non-null   str    
 2   Weekly_Sales  6435 non-null   float64
 3   Holiday_Flag  6435 non-null   int64  
 4   Temperature   6435 non-null   float64
 5   Fuel_Price    6435 non-null   float64
 6   CPI           6435 non-null   float64
 7   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), str(1)
memory usage: 465.2 KB
None

--- 2. Checking for Explicit Missing Values (NaN) ---
Store           0
Date            0
Weekly_Sales    0
Holiday_Flag    0
Temperature     0
Fuel_Price      0
CPI             0
Unemployment    0
dtype: int64

--- 3. Checking Data Range & Scale Disparities ---
                     min          mean           max            std
Store              1.000  2.300000e+

In [2]:
#task(2)
import pandas as pd
import numpy as np

# Load Walmart dataset
df = pd.read_csv("Walmart_Sales.csv")

# Convert Date to appropriate format as discovered in Task 1
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Replicating the assignment environment: Inject simulated missing row inputs for resilient pipeline testing
df_missing = df.copy()
df_missing.loc[0:5, 'Weekly_Sales'] = np.nan

# --- Applying Missing Value Strategy ---
df_imputed_median = df_missing.copy()
median_sales = df_imputed_median['Weekly_Sales'].median()

# Fill missing items using the calculated Median
df_imputed_median['Weekly_Sales'] = df_imputed_median['Weekly_Sales'].fillna(median_sales)

# Verify completeness
print("Remaining missing entries:", df_imputed_median['Weekly_Sales'].isna().sum())

Remaining missing entries: 0


In [3]:
#task(3)
# Calculate Quantiles and Interquartile Range (IQR) for Weekly_Sales
Q1 = df['Weekly_Sales'].quantile(0.25)
Q3 = df['Weekly_Sales'].quantile(0.75)
IQR = Q3 - Q1

# Establish upper and lower fences
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

# Filter data within acceptable non-anomalous statistical ranges
df_no_outliers = df[(df['Weekly_Sales'] >= lower_fence) & (df['Weekly_Sales'] <= upper_fence)]

print(f"Original Row Count: {df.shape[0]}")
print(f"Row Count After Trimming Outliers: {df_no_outliers.shape[0]}")
print(f"Total Outliers Dropped: {df.shape[0] - df_no_outliers.shape[0]}")

Original Row Count: 6435
Row Count After Trimming Outliers: 6401
Total Outliers Dropped: 34


In [4]:
#task(4)
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Select continuous indicators for processing
features_to_scale = ['Weekly_Sales', 'Fuel_Price']

In [5]:
#task(4)
scaler_minmax = MinMaxScaler()
df_minmax = df[features_to_scale].copy()

df_minmax[features_to_scale] = scaler_minmax.fit_transform(df_minmax)
print("--- Min-Max Scaled Output (Head) ---")
print(df_minmax.head())

--- Min-Max Scaled Output (Head) ---
   Weekly_Sales  Fuel_Price
0      0.397291    0.050100
1      0.396811    0.038076
2      0.388501    0.021042
3      0.332458    0.044589
4      0.372661    0.076653


In [6]:
#task(4)
scaler_zscore = StandardScaler()
df_zscore = df[features_to_scale].copy()

df_zscore[features_to_scale] = scaler_zscore.fit_transform(df_zscore)
print("\n--- Z-Score Standardized Output (Head) ---")
print(df_zscore.head())


--- Z-Score Standardized Output (Head) ---
   Weekly_Sales  Fuel_Price
0      1.057420   -1.713800
1      1.054348   -1.766089
2      1.001206   -1.840166
3      0.642828   -1.737766
4      0.899914   -1.598328


In [7]:
#task(5)
from sklearn.decomposition import PCA

# Prepare standardized multi-feature inputs to satisfy scaling assumptions for PCA
features_all_numeric = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
X_standardized = StandardScaler().fit_transform(df[features_all_numeric])

# Apply Principal Component Analysis across components
pca = PCA(n_components=len(features_all_numeric))
pca.fit(X_standardized)

# Print proportions
for i, variance in enumerate(pca.explained_variance_ratio_):
    print(f"Principal Component {i+1} (PC{i+1}): {variance:.4f} ({variance*100:.2f}%)")

Principal Component 1 (PC1): 0.2669 (26.69%)
Principal Component 2 (PC2): 0.2372 (23.72%)
Principal Component 3 (PC3): 0.2169 (21.69%)
Principal Component 4 (PC4): 0.1777 (17.77%)
Principal Component 5 (PC5): 0.1013 (10.13%)
